<a href="https://colab.research.google.com/github/smartinezai/smol-course/blob/smartinez-patch-1/%20%20%20%201_instruction_tuning%20%20/notebooks/chat_templates_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring Chat Templates with SmolLM2

This notebook demonstrates how to use chat templates with the `SmolLM2` model. Chat templates help structure interactions between users and AI models, ensuring consistent and contextually appropriate responses.

In [55]:
# Install the requirements in Google Colab
!pip install transformers datasets trl huggingface_hub

# Authenticate to Hugging Face
from huggingface_hub import login

login()

# for convenience you can create an environment variable containing your hub token as HF_TOKEN

In [56]:
# Import necessary libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import setup_chat_format
import torch

## SmolLM2 Chat Template

Let's explore how to use a chat template with the `SmolLM2` model. We'll define a simple conversation and apply the chat template.

In [57]:
# Dynamically set the device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)
model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

In [58]:
# Define messages for SmolLM2
messages = [
    {"role": "user", "content": "Hello, how are you?"},
    {
        "role": "assistant",
        "content": "I'm doing well, thank you! How can I assist you today?",
    },
]

# Apply chat template without tokenization

The tokenizer represents the conversation as a string with special tokens to describe the role of the user and the assistant.


In [59]:
type(messages)

list

In [60]:
input_text = tokenizer.apply_chat_template(messages, tokenize=False)

print("Conversation with template:", input_text)
print(type(input_text))

Conversation with template: <|im_start|>user
Hello, how are you?<|im_end|>
<|im_start|>assistant
I'm doing well, thank you! How can I assist you today?<|im_end|>

<class 'str'>


# Decode the conversation

Note that the conversation is represented as above but with a further assistant message.


In [61]:
input_text = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True
)

print("Conversation decoded:", tokenizer.decode(token_ids=input_text))

Conversation decoded: <|im_start|>user
Hello, how are you?<|im_end|>
<|im_start|>assistant
I'm doing well, thank you! How can I assist you today?<|im_end|>
<|im_start|>assistant



# Tokenize the conversation

Of course, the tokenizer also tokenizes the conversation and special token as ids that relate to the model's vocabulary.



In [62]:
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

print("Conversation tokenized:", input_text)

Conversation tokenized: [1, 4093, 198, 19556, 28, 638, 359, 346, 47, 2, 198, 1, 520, 9531, 198, 57, 5248, 2567, 876, 28, 9984, 346, 17, 1073, 416, 339, 4237, 346, 1834, 47, 2, 198, 1, 520, 9531, 198]


<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Exercise: Process a dataset for SFT</h2>
    <p>Take a dataset from the Hugging Face hub and process it for SFT. </p>
    <p><b>Difficulty Levels</b></p>
    <p>🐢 Convert the `HuggingFaceTB/smoltalk` dataset into chatml format.</p>
    <p>🐕 Convert the `openai/gsm8k` dataset into chatml format.</p>
</div>

In [63]:
from IPython.core.display import display, HTML

display(
    HTML(
        """<iframe
  src="https://huggingface.co/datasets/HuggingFaceTB/smoltalk/embed/viewer/all/train?row=0"
  frameborder="0"
  width="100%"
  height="360px"
></iframe>
"""
    )
)

In [104]:
from datasets import load_dataset

ds = load_dataset("HuggingFaceTB/smoltalk", "everyday-conversations")


def process_dataset(sample):
    # TODO: 🐢 Convert the sample into a chat format
    # use the tokenizer's method to apply the chat template
  #  print(f" sample [messages] type before formatting {type(sample['messages' ])}")


    sample["messages"] = tokenizer.apply_chat_template(
        sample["messages"],
        tokenize=False,
        add_generation_prompt=True)
   # print(f" type of sample after applying template {type(sample)}")
    print(sample)
    return sample


ds = ds.map(process_dataset)

Map:   0%|          | 0/2260 [00:00<?, ? examples/s]

{'full_topic': 'Travel/Vacation destinations/Beach resorts', 'messages': "<|im_start|>user\nHi there<|im_end|>\n<|im_start|>assistant\nHello! How can I help you today?<|im_end|>\n<|im_start|>user\nI'm looking for a beach resort for my next vacation. Can you recommend some popular ones?<|im_end|>\n<|im_start|>assistant\nSome popular beach resorts include Maui in Hawaii, the Maldives, and the Bahamas. They're known for their beautiful beaches and crystal-clear waters.<|im_end|>\n<|im_start|>user\nThat sounds great. Are there any resorts in the Caribbean that are good for families?<|im_end|>\n<|im_start|>assistant\nYes, the Turks and Caicos Islands and Barbados are excellent choices for family-friendly resorts in the Caribbean. They offer a range of activities and amenities suitable for all ages.<|im_end|>\n<|im_start|>user\nOkay, I'll look into those. Thanks for the recommendations!<|im_end|>\n<|im_start|>assistant\nYou're welcome. I hope you find the perfect resort for your vacation.<|i

Map:   0%|          | 0/119 [00:00<?, ? examples/s]

{'full_topic': 'Travel/Tourist attractions/Local markets', 'messages': "<|im_start|>user\nHey!<|im_end|>\n<|im_start|>assistant\nHello! How can I help you today?<|im_end|>\n<|im_start|>user\nI'm planning a trip to Paris. What are some popular tourist attractions?<|im_end|>\n<|im_start|>assistant\nThe Eiffel Tower, the Louvre Museum, and Notre Dame Cathedral are must-visit places in Paris.<|im_end|>\n<|im_start|>user\nThat sounds great. Are there any local markets I should check out?<|im_end|>\n<|im_start|>assistant\nYes, the Champs-Élysées Christmas Market and the Marché aux Puces de Saint-Ouen (flea market) are very popular among tourists and locals alike.<|im_end|>\n<|im_start|>user\nAwesome, thank you for the recommendations!<|im_end|>\n<|im_start|>assistant\nYou're welcome! Have a great time in Paris!<|im_end|>\n<|im_start|>assistant\n"}
{'full_topic': 'Technology/Cameras/Camera brands', 'messages': "<|im_start|>user\nHi there<|im_end|>\n<|im_start|>assistant\nHello! How can I help

In [66]:
display(
    HTML(
        """<iframe
  src="https://huggingface.co/datasets/openai/gsm8k/embed/viewer/main/train"
  frameborder="0"
  width="100%"
  height="360px"
></iframe>
"""
    )
)

In [105]:
ds = load_dataset("openai/gsm8k", "main")


def process_dataset(sample):
    # TODO: 🐕 Convert the sample into a chat format
    print("SAMPLE \BEFORE ANYJHING IS DONE")
    print(sample)

    exchange = [
    {"role": "user", "content": sample["question"]},
    {
        "role": "assistant",
        "content": sample["answer"]}]
    print(f"exchange thingy: {exchange}")
    sample["messages"] = tokenizer.apply_chat_template(
        exchange,
        tokenize=False,
        add_generation_prompt=True)
    # 1. create a message format with the role and content
    print("printing sample AFTER APPLY>ING TEMPOLATEA")
    print(sample)
    print(type(sample))
    print("DONE PRINTING SAMPLE")
    # 2. apply the chat template to the samples using the tokenizer's method

    return sample


ds = ds.map(process_dataset)

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Streaming output truncated to the last 5000 lines.
<class 'datasets.formatting.formatting.LazyRow'>
DONE PRINTING SAMPLE
SAMPLE \BEFORE ANYJHING IS DONE
{'question': 'Amaya is watching a movie that she finds kind of boring, so she keeps tuning out and having to rewind it to catch what she missed. She watches 35 minutes before she realizes she needs to rewind the movie to catch something she missed, a process that adds 5 minutes to her total viewing time. She watches the movie for another 45 minutes but has to rewind it again, adding 15 minutes to her total time. Finally, she watches the last 20 minutes uninterrupted. If she added up the duration of the film plus all the times she had to rewind and re-watch parts, how many minutes did it take her to watch the movie?', 'answer': 'Amaya watches the movie without rewinding for 20 minutes + 15 minutes + 45 minutes + 20 minutes = <<20+15+45+20=100>>100 minutes duration of the movie.\nHowever, Amaya also spent extra time re-watching parts she

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Streaming output truncated to the last 5000 lines.
<class 'datasets.formatting.formatting.LazyRow'>
DONE PRINTING SAMPLE
SAMPLE \BEFORE ANYJHING IS DONE
{'question': 'For four hours, Patrick sold 15 cups of lemonade per hour at a price of $0.50 per cup.  In the next two hours, he sold 10 cups of lemonade per hour at a price of $0.60 per cup.  How much money did Patrick earn, in dollars, from selling lemonade for 6 hours?', 'answer': 'Patrick earned $0.50 x 15 = $<<0.5*15=7.50>>7.50 from selling the lemonade for $0.50 per cup.\nSo for four hours, his total earnings are $7.50 x 4 = $<<7.5*4=30>>30.\nHe earned $0.60 x 10 = $<<0.60*10=6>>6 from selling the lemonade for $0.60 per cup.\nSo for two hours, his total earnings are $6 x 2 = $<<6*2=12>>12.\nTherefore, Patrick earn a total of $30 + $12 = $<<30+12=42>>42 from selling lemonade for 6 hours.\n#### 42'}
exchange thingy: [{'role': 'user', 'content': 'For four hours, Patrick sold 15 cups of lemonade per hour at a price of $0.50 per cup.  

## Conclusion

This notebook demonstrated how to apply chat templates to different models, `SmolLM2`. By structuring interactions with chat templates, we can ensure that AI models provide consistent and contextually relevant responses.

In the exercise you tried out converting a dataset into chatml format. Luckily, TRL will do this for you, but it's useful to understand what's going on under the hood.